# **RAG Application Using Type Sense**

In [21]:
import typesense
from dotenv import load_dotenv
load_dotenv()

True

**Making the client of typesense**

In [22]:
import os
typesense_host = os.environ["TYPESENSE_HOST"]
typesense_api_key = os.environ["TYPESENSE_API_KEY"]
client=typesense.Client({
  'nodes': [{
    'host': typesense_host,  # For Typesense Cloud use xxx.a1.typesense.net
    'port': '443',       # For Typesense Cloud use 443
    'protocol': 'https'    # For Typesense Cloud use https
  }],
  'api_key': typesense_api_key,
  'connection_timeout_seconds': 2
})

**Creating the collection**

In [24]:
books_schema = {
  'name': 'book',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}

print(client.collections.create(books_schema))

{'created_at': 1791270769, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

**Adding the Data to collection of typesense**

In [25]:
with open('books.jsonl', 'r', encoding='utf-8') as f:
    data = f.read()
    client.collections['books'].documents.import_(data)

**Search from the collection using query/retrieve from typesense**

In [26]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [27]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc",
    'filter_by':"publication_year:<2000"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 5,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' Ru

**Langchain + Typsense + Groq LLM + RAG Application**

In [28]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [29]:
import os
groq_api_key = os.environ["GROQ_API_KEY"]

In [30]:
loadder = TextLoader("text.txt")
documents = loadder.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
doc = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 377.11it/s]


In [32]:
import os
typesense_host = os.environ["TYPESENSE_HOST"]
typesense_api_key = os.environ["TYPESENSE_API_KEY"]

docsearch=Typesense.from_documents(
    doc,
    embeddings,
    typesense_client_params={
        "host": typesense_host,  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": typesense_api_key,
        "typesense_collection_name": "lang-chain"
    },
    
)

In [39]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs)
print("\n\n\n")
print(found_docs[0].page_content)

[Document(metadata={'source': 'text.txt'}, page_content='ARTIFICIAL INTELLIGENCE (AI) - COMPLETE STUDY NOTES\n=======================================================\n\nPurpose:\nThis text file is a structured reference for learning Artificial Intelligence\nfrom beginner to advanced level. It covers AI, Machine Learning, Deep Learning,\nNLP, Computer Vision, Generative AI, LLMs, RAG, Agents, MLOps, deployment,\nevaluation, safety, projects, and interview preparation.\n\n=======================================================\n1. WHAT IS ARTIFICIAL INTELLIGENCE?\n=======================================================\n\nArtificial Intelligence (AI) is the field of computer science focused on\nbuilding systems that can perform tasks that normally require human-like\nintelligence.\n\nExamples:\n- Understanding language\n- Recognizing images\n- Making predictions\n- Recommending products\n- Planning and decision-making\n- Generating text, images, audio, or code\n- Detecting anomalies\n- C

In [40]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001E62D476490>, search_kwargs={})

In [42]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

print(retriever.invoke(query)[0].page_content)

ARTIFICIAL INTELLIGENCE (AI) - COMPLETE STUDY NOTES

Purpose:
This text file is a structured reference for learning Artificial Intelligence
from beginner to advanced level. It covers AI, Machine Learning, Deep Learning,
NLP, Computer Vision, Generative AI, LLMs, RAG, Agents, MLOps, deployment,
evaluation, safety, projects, and interview preparation.

1. WHAT IS ARTIFICIAL INTELLIGENCE?

Artificial Intelligence (AI) is the field of computer science focused on
building systems that can perform tasks that normally require human-like
intelligence.

Examples:
- Understanding language
- Recognizing images
- Making predictions
- Recommending products
- Planning and decision-making
- Generating text, images, audio, or code
- Detecting anomalies
- Controlling robots
